In [ ]:
# # install octave
# !sudo apt-get -qq update
# !sudo apt-get -qq install octave octave-signal liboctave-dev

# # install oct2py that compatible with colab
# import os

# from pkg_resources import get_distribution

# os.system(
#     f"pip install -qq"
#     f" ipykernel=={get_distribution('ipykernel').version}"
#     f" ipython=={get_distribution('ipython').version}"
#     f" tornado=={get_distribution('tornado').version}"
#     f" oct2py"
# )

# # install packages
# !pip install -qq matpower matpowercaseframesa

In [ ]:
import oct2py

import matpower

print(f"oct2py version: {oct2py.__version__}")
print(f"matpower version: {matpower.__version__}")

oct2py version: 6.0.1
matpower version: 8.1.0.2.2.4


In [ ]:
import os

from matpowercaseframes import CaseFrames

from matpower import path_matpower, start_instance

In [ ]:
path = os.path.join(path_matpower, "data/case9.m")

In [ ]:
m = start_instance()

## Original Case

In [ ]:
cf = CaseFrames(path, load_case_engine=m)

mpc = m.runpf(cf.to_mpc(), verbose=False)
cf = CaseFrames(mpc)
cf.infer_numpy()

display(cf.branch)
display(cf.bus)
display(cf.gen)

,F_BUS,T_BUS,BR_R,BR_X,BR_B,RATE_A,RATE_B,RATE_C,TAP,SHIFT,BR_STATUS,ANGMIN,ANGMAX,PF,QF,PT,QT
branch,,,,,,,,,,,,,,,,,
1,1,4,0.0000,0.0576,0.000,250,250,250,0,0,1,-360,360,71.641021,27.045924,-71.641021,-23.923127
2,4,5,0.0170,0.0920,0.158,250,250,250,0,0,1,-360,360,30.703670,1.030006,-30.537263,-16.543365
3,5,6,0.0390,0.1700,0.358,150,150,150,0,0,1,-360,360,-59.462737,-13.456635,60.816586,-18.074836
4,3,6,0.0000,0.0586,0.000,300,300,300,0,0,1,-360,360,85.000000,-10.859709,-85.000000,14.955327
5,6,7,0.0119,0.1008,0.209,150,150,150,0,0,1,-360,360,24.183414,3.119508,-24.095417,-24.295823
6,7,8,0.0085,0.0720,0.149,250,250,250,0,0,1,-360,360,-75.904583,-10.704177,76.379866,-0.797331
7,8,2,0.0000,0.0625,0.000,250,250,250,0,0,1,-360,360,-163.000000,9.178149,163.000000,6.653660
8,8,9,0.0320,0.1610,0.306,250,250,250,0,0,1,-360,360,86.620134,-8.380817,-84.320163,-11.312751
9,9,4,0.0100,0.0850,0.176,250,250,250,0,0,1,-360,360,-40.679837,-38.687249,40.937352,22.893121


,BUS_I,BUS_TYPE,PD,QD,GS,BS,BUS_AREA,VM,VA,BASE_KV,ZONE,VMAX,VMIN
bus,,,,,,,,,,,,,
1,1,3,0,0,0,0,1,1.040000,0.000000,345,1,1.1,0.9
2,2,2,0,0,0,0,1,1.025000,9.280005,345,1,1.1,0.9
3,3,2,0,0,0,0,1,1.025000,4.664751,345,1,1.1,0.9
4,4,1,0,0,0,0,1,1.025788,-2.216788,345,1,1.1,0.9
5,5,1,90,30,0,0,1,1.012654,-3.687396,345,1,1.1,0.9
6,6,1,0,0,0,0,1,1.032353,1.966716,345,1,1.1,0.9
7,7,1,100,35,0,0,1,1.015883,0.727536,345,1,1.1,0.9
8,8,1,0,0,0,0,1,1.025769,3.719701,345,1,1.1,0.9
9,9,1,125,50,0,0,1,0.995631,-3.988805,345,1,1.1,0.9


,GEN_BUS,PG,QG,QMAX,QMIN,VG,MBASE,GEN_STATUS,PMAX,PMIN,...,PC2,QC1MIN,QC1MAX,QC2MIN,QC2MAX,RAMP_AGC,RAMP_10,RAMP_30,RAMP_Q,APF
gen,,,,,,,,,,,,,,,,,,,,,
1,1,71.641021,27.045924,300,-300,1.040,100,1,250,10,...,0,0,0,0,0,0,0,0,0,0
2,2,163.000000,6.653660,300,-300,1.025,100,1,300,10,...,0,0,0,0,0,0,0,0,0,0
3,3,85.000000,-10.859709,300,-300,1.025,100,1,270,10,...,0,0,0,0,0,0,0,0,0,0


In [ ]:
tripped_lines = [3, 8]
cf.branch.loc[tripped_lines, "BR_STATUS"] = 0

In [ ]:
[groups, isolated] = m.find_islands(cf.to_mpc(), nout=2)
groups, isolated

(Cell([[array([[2., 3., 6., 7., 8.]]), array([[1., 4., 5., 9.]])]],
       dtype=object),
 array([], shape=(1, 0), dtype=float64))

In [ ]:
mpcs = m.extract_islands(cf.to_mpc(), groups)

cfs = {}
for isl, mpc in enumerate(mpcs.flatten()):
    print(f"Island {isl}")
    cfs[isl] = CaseFrames(mpc)
    cfs[isl].infer_numpy()

    display(cfs[isl].branch[["F_BUS", "T_BUS", "BR_STATUS"]])
    display(cfs[isl].bus)
    display(cfs[isl].gen)

Island 0


,F_BUS,T_BUS,BR_STATUS
branch,,,
1,3,6,1
2,6,7,1
3,7,8,1
4,8,2,1


,BUS_I,BUS_TYPE,PD,QD,GS,BS,BUS_AREA,VM,VA,BASE_KV,ZONE,VMAX,VMIN
bus,,,,,,,,,,,,,
2,2,2,0,0,0,0,1,1.025000,9.280005,345,1,1.1,0.9
3,3,2,0,0,0,0,1,1.025000,4.664751,345,1,1.1,0.9
6,6,1,0,0,0,0,1,1.032353,1.966716,345,1,1.1,0.9
7,7,1,100,35,0,0,1,1.015883,0.727536,345,1,1.1,0.9
8,8,1,0,0,0,0,1,1.025769,3.719701,345,1,1.1,0.9


,GEN_BUS,PG,QG,QMAX,QMIN,VG,MBASE,GEN_STATUS,PMAX,PMIN,...,PC2,QC1MIN,QC1MAX,QC2MIN,QC2MAX,RAMP_AGC,RAMP_10,RAMP_30,RAMP_Q,APF
gen,,,,,,,,,,,,,,,,,,,,,
1,2,163,6.653660,300,-300,1.025,100,1,300,10,...,0,0,0,0,0,0,0,0,0,0
2,3,85,-10.859709,300,-300,1.025,100,1,270,10,...,0,0,0,0,0,0,0,0,0,0


Island 1


,F_BUS,T_BUS,BR_STATUS
branch,,,
1,1,4,1
2,4,5,1
3,9,4,1


,BUS_I,BUS_TYPE,PD,QD,GS,BS,BUS_AREA,VM,VA,BASE_KV,ZONE,VMAX,VMIN
bus,,,,,,,,,,,,,
1,1,3,0,0,0,0,1,1.040000,0.000000,345,1,1.1,0.9
4,4,1,0,0,0,0,1,1.025788,-2.216788,345,1,1.1,0.9
5,5,1,90,30,0,0,1,1.012654,-3.687396,345,1,1.1,0.9
9,9,1,125,50,0,0,1,0.995631,-3.988805,345,1,1.1,0.9


,GEN_BUS,PG,QG,QMAX,QMIN,VG,MBASE,GEN_STATUS,PMAX,PMIN,...,PC2,QC1MIN,QC1MAX,QC2MIN,QC2MAX,RAMP_AGC,RAMP_10,RAMP_30,RAMP_Q,APF
gen,,,,,,,,,,,,,,,,,,,,,
1,1,71.641021,27.045924,300,-300,1.04,100,1,250,10,...,0,0,0,0,0,0,0,0,0,0
